In [6]:
import re
import json
from pathlib import Path
from pprint import pprint
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter, 
    MarkdownHeaderTextSplitter
)

load_dotenv()

RAW_DIR = Path("../../data/raw/confluence")
PROCESSED_DIR = Path("../../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

### `extract_front_matter` — Implementation Details

This function parses **structured metadata** from the top of a markdown file and returns it as a `dict`.

#### Why we need it

Our raw markdown docs contain metadata like title, version, author, and dates — but they're embedded in the document text, not in a separate config. We need to extract these into structured fields so we can attach them as **metadata** to each chunk later.

#### Three patterns it handles

| # | Pattern | Example | Regex |
|---|---------|---------|-------|
| 1 | **H1 title** — first `# Heading` in the file | `# AMS Admin Tool` | `^#\s+(.+)$` |
| 2 | **Blockquote key-value** — `> **Key:** Value` | `> **Version:** 1.0` | `>\s*\*\*(.+?):\*\*\s*(.+)` |
| 3 | **Bold key-value** (no blockquote) — `**Key:** Value` | `**Last updated:** 2026-03-18` | `(?<!>)\s*\*\*(.+?):\*\*\s*(.+)` |

#### Step-by-step walkthrough

1. **Extract title** — `re.search` finds the **first** H1 heading (`# ...`) anywhere in the text and stores it as `doc_title`.

2. **Extract blockquote fields** — `re.finditer` loops over all `> **Key:** Value` lines. Each key is lowercased and spaces replaced with underscores (e.g., `Last Updated` → `last_updated`).

3. **Extract bold key-value fields** — Same pattern but **without** the `>` prefix. The `(?<!>)` is a **negative lookbehind** that ensures we don't double-match blockquote lines. The `if key not in meta` check prevents overwriting values already captured in step 2.

#### Key regex concepts used

- `re.search()` — finds the **first** match (used for title)
- `re.finditer()` — finds **all** matches, returns an iterator (used for key-value pairs)
- `re.MULTILINE` — makes `^` and `$` match start/end of **each line**, not just the whole string
- `(?<!>)` — **negative lookbehind**: match only if NOT preceded by `>`
- `(.+?)` — **non-greedy** capture: matches as few characters as possible (stops at the first `:`)

In [2]:
def extract_front_matter(text: str) -> dict:
    """Parse structured fields from the top of a markdown file.

    Handles two common patterns in our docs:
      1.  # Title on line 1
      2.  > **Key:** Value   (blockquote front matter)
      3.  **Key:** Value     (bold key-value on early lines)
    """
    meta = {}

    # 1) Document title — first H1
    title_match = re.search(r"^#\s+(.+)$", text, re.MULTILINE)
    if title_match:
        meta["doc_title"] = title_match.group(1).strip()

    # 2) Blockquote front matter:  > **Version:** 1.0
    for m in re.finditer(r">\s*\*\*(.+?):\*\*\s*(.+)", text):
        key = m.group(1).strip().lower().replace(" ", "_")
        meta[key] = m.group(2).strip()

    # 3) Bold key-value (non-blockquote):  **Last updated:** 2026-03-18
    for m in re.finditer(r"(?<!>)\s*\*\*(.+?):\*\*\s*(.+)", text):
        key = m.group(1).strip().lower().replace(" ", "_")
        if key not in meta:  # don't overwrite blockquote matches
            meta[key] = m.group(2).strip()

    return meta

In [7]:
# ── Derive doc_type from folder path ────────────────────────────────────────

DOC_TYPE_MAP = {
    "architecture": "architecture",
    "features": "feature",
    "operations": "operations",
    "business": "business",
}

def derive_doc_type(file_path: Path, base_dir: Path) -> str:
    """Map folder name to a doc_type label."""
    relative = file_path.relative_to(base_dir)
    parts = relative.parts  # e.g. ('architecture', 'overview.md')
    if len(parts) >= 2:
        folder = parts[0]
        return DOC_TYPE_MAP.get(folder, "general")
    return "index"  # top-level files like index.md


# ── Quick test ───────────────────────────────────────────────────────────────

sample_path = RAW_DIR / "architecture" / "overview.md"
sample_text = sample_path.read_text()

print("=== Extracted front matter ===")
fm = extract_front_matter(sample_text)
pprint(fm)

print(f"\n=== Derived doc_type ===")
print(derive_doc_type(sample_path, RAW_DIR))

=== Extracted front matter ===
{'doc_title': 'AMS Admin Tool — Architecture Documentation',
 'last_updated': 'March 2026',
 'owner': 'Admin Tool Engineering',
 'status': 'Production',
 'version': '1.0'}

=== Derived doc_type ===
architecture


## Step 2 -- Heading-Aware Markdown Splitting

In [17]:
HEADERS_TO_SPLIT_ON = [
    ("#", "h1"),
    ("##", "h2"),
    ("###", "h3"),
]

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=HEADERS_TO_SPLIT_ON,
    strip_headers=False
)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, 
    chunk_overlap=CHUNK_OVERLAP
)

def split_markdown_file(file_path: Path, base_dir: Path) -> list[Document]:
    raw_text = file_path.read_text(encoding="utf-8")
    stat = file_path.stat()
    front_matter = extract_front_matter(raw_text)
    doc_type = derive_doc_type(file_path, base_dir)
    source = str(file_path.relative_to(base_dir))

    doc_meta = {
        "source": source,
        "doc_title": front_matter.get("doc_title", file_path.stem),
        "doc_type": doc_type,
        "audience": front_matter.get("audience", "all").lower(),
        "owner": front_matter.get("owner", ""),
        "status": front_matter.get("status", "").lower(),
        "version": front_matter.get("version", ""),
        "last_modified": front_matter.get(
            "last_updated",
            # fallback to file mtime
            __import__("datetime").datetime.fromtimestamp(stat.st_mtime).strftime("%Y-%m-%d"),
        ),
    }

    header_chunks = md_splitter.split_text(raw_text)
    final_chunks = text_splitter.split_documents(header_chunks)
    enriched = []
    for i, chunk in enumerate(final_chunks):
        heading_hierarchy = []
        for level in ["h1", "h2", "h3"]:
            if level in chunk.metadata:
                heading_hierarchy.append(chunk.metadata[level])
        # The most specific heading is the "section"
        section = heading_hierarchy[-1] if heading_hierarchy else ""
        
        chunk.metadata.update({
            **doc_meta,
            "section": section,
            "heading_hierarchy": heading_hierarchy,
            "chunk_index": i,
        })
        enriched.append(chunk)
    return enriched


In [18]:
# Quick test on one file
test_chunks = split_markdown_file(RAW_DIR / "architecture" / "overview.md", RAW_DIR)
print(f"Chunks from overview.md: {len(test_chunks)}")
print(f"\n=== Sample chunk (index 3) ===")
print(f"Content preview: {test_chunks[3].page_content[:200]}...")
print(f"\nMetadata:")
pprint(test_chunks[3].metadata)

Chunks from overview.md: 40

=== Sample chunk (index 3) ===
Content preview: | **Canvas** | React Flow (@xyflow/react) | 12.10 | Visual node-and-edge flow graph |
| **Layout** | dagre (@dagrejs/dagre) | 2.0 | Automatic graph layout |
| **Code Editor** | CodeMirror 6 | 6.0 | Gr...

Metadata:
{'audience': 'all',
 'chunk_index': 3,
 'doc_title': 'AMS Admin Tool — Architecture Documentation',
 'doc_type': 'architecture',
 'h1': 'AMS Admin Tool — Architecture Documentation',
 'h2': 'Tech Stack',
 'heading_hierarchy': ['AMS Admin Tool — Architecture Documentation',
                       'Tech Stack'],
 'last_modified': 'March 2026',
 'owner': 'Admin Tool Engineering',
 'section': 'Tech Stack',
 'source': 'architecture/overview.md',
 'status': 'production',
 'version': '1.0'}


In [19]:
# ── Process all markdown files ────────────────────────────────────────────────

all_chunks: list[Document] = []

md_files = sorted(RAW_DIR.rglob("*.md"))
print(f"Found {len(md_files)} markdown files\n")

for file_path in md_files:
    chunks = split_markdown_file(file_path, RAW_DIR)
    all_chunks.extend(chunks)
    print(f"  {str(file_path.relative_to(RAW_DIR)):<50} → {len(chunks):>3} chunks")

print(f"\n{'='*60}")
print(f"Total chunks: {len(all_chunks)}")

Found 10 markdown files

  architecture/code-quality.md                       →  28 chunks
  architecture/firebase.md                           →  32 chunks
  architecture/overview.md                           →  40 chunks
  architecture/redux.md                              →  20 chunks
  architecture/theming.md                            →  23 chunks
  features/action-builder.md                         →  41 chunks
  features/history-service.md                        →  38 chunks
  features/journey-canvas.md                         →  39 chunks
  index.md                                           →   7 chunks
  operations/code-quality-audit-plan.md              →  24 chunks

Total chunks: 292
